# SAC arrival_v2 — history k=12 cross-seed sister (seed=0) on single_cross_s0 (1M, vanilla)

**Pre-context（commit `352ed78` + k=12 seed=42 PASS-PLATEAU 回流 + k=8 multi-seed 3-anchor 回流 2026-05-19）**：[`docs/arrival_v2_experiment_report.md`](../docs/arrival_v2_experiment_report.md) 当前 single_cross_s0 vanilla SAC 完整画像（6 runs）：

| run                                | seed | k    | final  | peak@step    | mean39 | OOB    | n_succ | Gate         |
|---                                 |---:  |---:  |---:    |---           |---:    |---:    |---:    |---           |
| §7.6.4 s0_k4                       | 42   | 4    | 0.100  | 0.367 @ 975k | 0.221  | 0.667  | 35/39  | FAIL floor   |
| §7.7.1 s0_k4 sister                | 0    | 4    | 0.400  | 0.533 @ 625k | 0.218  | 0.200  | 34/39  | FAIL         |
| **§7.8 s0_k8 anchor**              | 42   | 8    | **0.900** | 0.900 @ 475k | **0.636** | **0.100** | 37/39 | **PASS 5/5** |
| §7.8' s0_k8                        | 0    | 8    | 0.500  | 0.500 @ 925k | 0.260  | 0.133  | 32/39  | PARTIAL 2/5  |
| §7.8'' s0_k8                       | 7    | 8    | 0.867  | 0.900 @ 550k | 0.518  | 0.133  | 31/39  | BORDER 4/5   |
| **§8P1#2 s0_k12 anchor**           | 42   | 12   | **0.900** | 0.900 @ **375k** | 0.652 | **0.100** | 35/39 | **PASS 5/5 PLATEAU** |
| **本 notebook s0_k12 sister**     | **0**| 12   | ?      | ?            | ?      | ?      | ?      | ?            |

**关键 open questions（本 notebook 必须区分）**：

| Hypothesis | 含义 |
|---|---|
| **H_rescue: k=12 真有 + monotonic 增益，k=8 seed=0 仅是窗口不够长** | k=12 s0 应 5/5 PASS（或 BORDER），与 k=12 s42 类似 |
| **H_seed-stall: seed=0 有 init-dep local minimum，与 k 无关** | k=12 s0 应仍 PARTIAL（final ≈ 0.5），与 k=8 s0 类似 |
| **H_overstale: k=12 在 seed=0 上 over-stale，反害** | k=12 s0 final < k=8 s0 final |
| **H_collapse: severe regression** | k=12 s0 final < 0.4，需 audit |

**本 notebook 任务（pure vanilla + history k=12，单变量 seed swap from k=12 anchor）**：

| 维度                  | §8P1#2 anchor (k=12, seed=42) | 本 notebook (k=12, seed=0) |
|---                    |---                            |---                          |
| `--seed`              | 42                            | **0** ← 唯一变量            |
| `--history-length`    | 12                            | 12                          |
| algorithm             | vanilla SAC                   | vanilla SAC                 |
| sensor layout         | s0 (DVL-only)                 | s0 (DVL-only)               |
| reward                | arrival_v2                    | arrival_v2                  |
| flow U / target       | 1.5 / 1.5                     | 1.5 / 1.5                   |
| total_steps           | 1M                            | 1M                          |
| num_envs              | 6                             | 6                           |
| benchmark             | `single_u15_cross_tgt15`      | 同                          |
| obs_dim               | 144 (12×12)                   | 144                         |

**Seed 选择 = 0 的理由**：与 k=4 sister (§7.7.1) + k=8 sister (§7.8') seed 对齐，构成完整 `k ∈ {4, 8, 12} × seed=0` 三元 paired contrast。

**Gate**（与 §7 / §7.6 / §7.7 / §7.8 / §8P1#2 同口径）：
- `final_success_rate ≥ 0.85`
- `last100k_mean ≥ 0.9 × peak`
- `final_oob_rate ≤ 0.10`
- `include_episode_context_obs == True`
- `timeout_bootstrap_semantics == 'terminal'`

**Cross-seed monotonicity verdict 规则（5-tier，吸取 build_k8_seed7.py 4-tier BORDERLINE 漏覆盖 bug 教训）**：

| Verdict | 触发条件 | §7.8 / §8P1#2 主张含义 |
|---|---|---|
| **CROSS-SEED-RESCUE** | k=12 s0 strict 5/5 PASS（final≥0.85 AND OOB≤0.10）| 强证据 H_rescue。§7.8 主张升格「k=4→8 closes gap on lucky seeds; k=8→12 进一步 rescue unlucky seeds」；§8 P1#2 升格 monotonic claim |
| **CROSS-SEED-BORDERLINE-PASS** | final≥0.85 AND 0.10 < OOB ≤ 0.135 | final 维度 H_rescue 但 OOB 仍 seed-dep（与 §7.8'' seed=7 BORDER 一致）；说明 SAC variance 主要表现在 OOB 末端而非 final goal 率；motivates §8 P0 variance reduction 但 history 路径仍 viable |
| **PERSISTENT-STALL** | 0.50 ≤ final < 0.85 AND \|final − k=8 s0 final\| ≤ 0.20 | 强证据 H_seed-stall。seed=0 有 init-dep local minimum 与 history length 无关；§8 P0 (DroQ / SimBa / N-Step) variance reduction 必要 |
| **UNEXPECTED-DECAY** | k=12 s0 final < k=8 s0 final − 0.05 OR 0.40 ≤ final < 0.50 | H_overstale 部分支持；over-stale history 在 seed=0 上反害；§8 P1#2 主张回退「k=8 sweet spot」 |
| **COLLAPSE** | final < 0.40 | H_collapse。严重下降；audit trainer_state（grad-norm / loss curve / obs_dim）排除 training bug |

**输出根（与 §8P1#2 k=12 anchor / §7.8' k=8 seed=0 互不覆盖）**：
- `experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_0/`

**总预算**：~2.5h L4（1 Colab Pro+ session；obs_dim 144 vs 96 增加 ~50% forward cost，但 hidden_dim 仍 256，主要瓶颈是 env step）。

**风格**：训练用 `!python -u -m scripts.train_sac` 直跑，与 §7.8 / §8P1#2 一致。


## 0. GPU sanity


In [ ]:
!nvidia-smi | head -10


## 1. Mount Drive + cwd


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR


## 2. Config — single phase（vanilla SAC + history k=12 + seed=0，与 §8P1#2 k=12 anchor 仅差一个 flag value）


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

# ==== SAC / env config (与 §8P1#2 k=12 anchor 严格一致，除 seed 外) ====
OBJECTIVE = 'arrival_v2'
PROBE_LAYOUT = 's0'
HISTORY_LENGTH = 12
TARGET_SPEED = 1.5
SEED = 0                                # ← 唯一与 §8P1#2 k=12 anchor (seed=42) 不同；cross-seed sister

RANDOM_STEPS = 5_000
UPDATE_AFTER = 5_000
BATCH_SIZE = 256
HIDDEN_DIM = 256
NUM_ENVS = 6
EVAL_EVERY = 25_000
EVAL_EPISODES = 30
CHECKPOINT_EVERY = 100_000
DEVICE = 'cuda'

# 显式拒绝所有 SAC 改进项 — 与 §7.8 / §8P1#2 一致
USE_ASYMMETRIC_CRITIC = False
USE_LAYERNORM = False
UPDATES_PER_STEP = 1
DROPOUT_RATE = 0.0

PASS_FINAL_SUCCESS = 0.85
PASS_LAST100_RATIO = 0.90
PASS_OOB_RATE = 0.10
BORDERLINE_OOB_RATE = 0.135  # BORDERLINE-PASS tier 上限

# Flow file（与 §7 / §8P1#2 全套严格一致）
SINGLE_FLOW = 'wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'

# ==== Single phase — single_cross s0 k=12 seed=0 (X_*) ====
X_BENCHMARK_KEY = 'single_u15_cross_tgt15'
X_TASK_GEOMETRY = 'cross_stream'
X_FLOW_PATH = SINGLE_FLOW
X_TOTAL_STEPS = 1_000_000
X_RUN_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_0')
X_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_0')
X_MANIFEST_PATH = Path(f'benchmarks/{X_BENCHMARK_KEY}.json')

# Baselines（cross-seed monotonicity 需要全 7-run 对照）
X_K12_S42_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_42')  # §8P1#2 anchor PASS
X_K8_S42_ROOT  = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_42')   # §7.8 anchor PASS
X_K8_S0_ROOT   = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_0')    # §7.8' PARTIAL (主对照 — same seed)
X_K8_S7_ROOT   = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_7')    # §7.8'' BORDERLINE-PASS
X_K4_S42_ROOT  = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')   # §7.6.4 floor
X_K4_S0_ROOT   = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_0')    # §7.7.1 sister
X_S1_UPPER_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')  # §7.1 upper

os.environ['PYTHONUNBUFFERED'] = '1'

print(f'PROBE_LAYOUT          = {PROBE_LAYOUT}')
print(f'OBJECTIVE             = {OBJECTIVE}')
print(f'HISTORY_LENGTH        = {HISTORY_LENGTH}')
print(f'SEED                  = {SEED}        ← cross-seed sister; 与 §8P1#2 anchor (seed=42) 不同')
print(f'NUM_ENVS              = {NUM_ENVS}')
print(f'USE_ASYMMETRIC_CRITIC = {USE_ASYMMETRIC_CRITIC}')
print(f'USE_LAYERNORM         = {USE_LAYERNORM}')
print(f'UPDATES_PER_STEP      = {UPDATES_PER_STEP}')
print(f'DROPOUT_RATE          = {DROPOUT_RATE}')
print()
print(f'benchmark             : {X_BENCHMARK_KEY}')
print(f'geometry              : {X_TASK_GEOMETRY}')
print(f'total_steps           : {X_TOTAL_STEPS:,}')
print(f'expected obs_dim      : 12 * {HISTORY_LENGTH} = {12 * HISTORY_LENGTH} (per train_config.txt across all baselines)')
print(f'run_root              : {X_RUN_ROOT}')
print(f'§8P1#2 k=12 s42 PASS  : {X_K12_S42_ROOT}')
print(f'§7.8   k=8  s42 PASS  : {X_K8_S42_ROOT}')
print(f"§7.8'  k=8  s0  PART  : {X_K8_S0_ROOT}")
print(f"§7.8'' k=8  s7  BORDER: {X_K8_S7_ROOT}")
print(f'§7.6.4 k=4  s42 floor : {X_K4_S42_ROOT}')
print(f'§7.7.1 k=4  s0  sister: {X_K4_S0_ROOT}')
print(f'§7.1   s1 k4 s42 upper: {X_S1_UPPER_ROOT}')


## 3. Preflight — flow / arrival_v2 candidate gate / reward unit tests / manifest / baseline 就位


In [ ]:
# Flow file
fp = Path(X_FLOW_PATH)
if not fp.exists():
    raise FileNotFoundError(f'missing flow file: {fp}')
print(f'[OK] flow file: {fp}  ({fp.stat().st_size / 1e6:.1f} MB)')


In [ ]:
!python -u -m scripts.validate_arrival_v2_candidate


In [ ]:
!python -u -m pytest tests/test_reward_objective.py -q


In [ ]:
if not X_MANIFEST_PATH.exists():
    !python -u -m scripts.generate_standard_benchmarks --benchmarks {X_BENCHMARK_KEY} --episodes {EVAL_EPISODES}
if not X_MANIFEST_PATH.exists():
    raise FileNotFoundError(f'manifest not generated: {X_MANIFEST_PATH}')
print(f'[OK] manifest ready: {X_MANIFEST_PATH}')


In [ ]:
# 检查 7 个对比 baseline 是否就位
for label, root, ref_final, ref_oob in [
    ('§8P1#2 k12 s42 PASS-PLAT  ', X_K12_S42_ROOT, 0.900, 0.100),
    ('§7.8   k8  s42 PASS       ', X_K8_S42_ROOT,  0.900, 0.100),
    ("§7.8'  k8  s0  PARTIAL    ", X_K8_S0_ROOT,   0.500, 0.133),
    ("§7.8'' k8  s7  BORDER     ", X_K8_S7_ROOT,   0.867, 0.133),
    ('§7.6.4 k4  s42 floor      ', X_K4_S42_ROOT,  0.100, 0.667),
    ('§7.7.1 k4  s0  sister     ', X_K4_S0_ROOT,   0.400, 0.200),
    ('§7.1   s1  k4 upper       ', X_S1_UPPER_ROOT, 0.900, 0.100),
]:
    fp = root / 'results' / 'final_eval.json'
    if fp.exists():
        d = json.loads(fp.read_text(encoding='utf-8'))
        f = float(d['eval_success_rate'])
        c = d.get('eval_termination_counts', {})
        n = float(d.get('num_eval_episodes', EVAL_EPISODES))
        oob = float(c.get('out_of_bounds', 0)) / max(n, 1.0)
        match = '✓' if abs(f - ref_final) < 0.05 and abs(oob - ref_oob) < 0.05 else '✗ mismatch'
        print(f'[OK] {label}: final={f:.4f}  oob={oob:.4f}  counts={c}  {match}')
    else:
        print(f'[WARN] {label}: {fp} 不存在 (后续 §5 diff 会回退到 report 转载值)')


## 4. Train — single_cross_s0 + history k=12 + seed=0 (1.0M, skip/resume)


In [ ]:
x_state_path = X_RUN_ROOT / 'trainer_state.json'
if x_state_path.exists():
    x_state = json.loads(x_state_path.read_text(encoding='utf-8'))
    x_current_step = int(x_state.get('env_step', 0))
else:
    x_current_step = 0
print(f'[state] X env_step = {x_current_step:,} / target {X_TOTAL_STEPS:,}')

if x_current_step >= X_TOTAL_STEPS:
    print(f'[skip] X already trained to {x_current_step:,} >= {X_TOTAL_STEPS:,}')
elif x_current_step > 0:
    print(f'[resume] X continuing from {x_current_step:,} -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {str(X_RUN_ROOT)} \
        --total-steps {X_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --device {DEVICE}
else:
    print(f'[train] X fresh start -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {X_FLOW_PATH} \
        --task-geometry {X_TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {X_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {str(X_RUN_ROOT)} \
        --checkpoint-dir {str(X_CKPT_ROOT)}


## 5. Summary + gate


In [ ]:
def summarize_phase(run_root: Path, total_steps: int, label: str, gate_filename: str):
    eval_log_path = run_root / 'results' / 'eval_log.csv'
    final_eval_path = run_root / 'results' / 'final_eval.json'
    trainer_state_path = run_root / 'trainer_state.json'
    train_config_path = run_root / 'results' / 'train_config.txt'

    if not eval_log_path.exists():
        raise FileNotFoundError(f'missing eval log: {eval_log_path}')
    if not final_eval_path.exists():
        raise FileNotFoundError(f'missing final eval: {final_eval_path}')

    df = pd.read_csv(eval_log_path)
    final_eval = json.loads(final_eval_path.read_text(encoding='utf-8'))
    trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8')) if trainer_state_path.exists() else {}

    history_from_config = 'NA'
    obs_dim_from_config = 'NA'
    if train_config_path.exists():
        for ln in train_config_path.read_text(encoding='utf-8').splitlines():
            s = ln.strip()
            if s.startswith('history_length='):
                history_from_config = s.split('=', 1)[1]
            elif s.startswith('obs_dim='):
                obs_dim_from_config = s.split('=', 1)[1]

    peak_success = float(df['eval_success_rate'].max()) if len(df) else 0.0
    peak_step = int(df.loc[df['eval_success_rate'].idxmax(), 'env_step']) if len(df) else 0
    last100 = df[df['env_step'] >= total_steps - 100_000].copy()
    last100_mean = float(last100['eval_success_rate'].mean()) if len(last100) else 0.0
    mean_all = float(df['eval_success_rate'].mean()) if len(df) else 0.0
    n_evals_with_success = int((df['eval_success_rate'] > 0).sum()) if len(df) else 0
    final_success = float(final_eval['eval_success_rate'])
    counts = final_eval.get('eval_termination_counts', {})
    num_eps = float(final_eval.get('num_eval_episodes', EVAL_EPISODES))
    oob_rate = float(counts.get('out_of_bounds', 0)) / max(num_eps, 1.0)

    print('=' * 100)
    print(f'{label}  (arrival_v2 / s0 / k=12 / seed={SEED} / vanilla / {total_steps:,} steps)')
    print('-' * 100)
    print(f"  final_success_rate    : {final_success:.4f}   gate >= {PASS_FINAL_SUCCESS:.2f}")
    print(f"  peak_success_rate     : {peak_success:.4f}   @ {peak_step:,}")
    print(f"  last100k_mean_success : {last100_mean:.4f}   gate >= {PASS_LAST100_RATIO * peak_success:.4f}")
    print(f"  full-traj mean        : {mean_all:.4f}   (39 evals)")
    print(f"  n_evals_with_success  : {n_evals_with_success} / {len(df)}")
    print(f"  final_oob_rate        : {oob_rate:.4f}   gate <= {PASS_OOB_RATE:.2f}")
    print(f"  obs_dim               : {obs_dim_from_config}   (expect 12*12=144)")
    print(f"  history_length        : {history_from_config}   (from train_config.txt)")
    print(f"  context_obs           : {trainer_state.get('include_episode_context_obs', 'NA')}")
    print(f"  timeout_bootstrap     : {trainer_state.get('timeout_bootstrap_semantics', 'NA')}")
    print(f"  termination           : {counts}")
    print('=' * 100)

    if len(df):
        print()
        print('[last 16 eval rows]')
        cols = ['env_step', 'eval_success_rate', 'eval_return', 'eval_safety_cost',
                'eval_time_s', 'eval_progress_ratio']
        available = [c for c in cols if c in df.columns]
        print(df[available].tail(16).to_string(index=False))

    checks = [
        ('final success >= 0.85', final_success >= PASS_FINAL_SUCCESS, f'{final_success:.4f}'),
        ('last100k mean >= 0.9 * peak', last100_mean >= PASS_LAST100_RATIO * peak_success,
         f'{last100_mean:.4f} / peak={peak_success:.4f}'),
        ('final OOB rate <= 0.10', oob_rate <= PASS_OOB_RATE, f'{oob_rate:.4f}'),
        ('arrival_v2 context obs enabled',
         trainer_state.get('include_episode_context_obs') is True,
         str(trainer_state.get('include_episode_context_obs'))),
        ('arrival_v2 timeout terminal semantics',
         trainer_state.get('timeout_bootstrap_semantics') == 'terminal',
         str(trainer_state.get('timeout_bootstrap_semantics'))),
    ]

    print()
    print('=' * 96)
    print(f"{'check':<48}{'pass':>8}{'detail':>40}")
    print('-' * 96)
    all_pass = True
    for name, ok, detail in checks:
        mark = 'PASS' if ok else 'FAIL'
        if not ok:
            all_pass = False
        print(f'{name:<48}{mark:>8}{detail:>40}')
    print('=' * 96)

    summary = {
        'phase': label,
        'objective': OBJECTIVE,
        'probe_layout': PROBE_LAYOUT,
        'history_length': int(history_from_config) if history_from_config != 'NA' else HISTORY_LENGTH,
        'seed': SEED,
        'total_steps': total_steps,
        'algorithm': 'sac_vanilla',
        'final_success_rate': final_success,
        'peak_success_rate': peak_success,
        'peak_step': peak_step,
        'last100k_mean_success': last100_mean,
        'mean_success_full_trajectory': mean_all,
        'n_evals_with_success': n_evals_with_success,
        'n_evals_total': len(df),
        'final_oob_rate': oob_rate,
        'termination_counts': counts,
        'all_pass': bool(all_pass),
        'checks': [{'name': n, 'ok': bool(ok), 'detail': d} for n, ok, d in checks],
    }
    out_path = run_root / 'results' / gate_filename
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f'[saved] {out_path}')
    return summary

x_summary = summarize_phase(
    X_RUN_ROOT,
    X_TOTAL_STEPS,
    'SINGLE_CROSS_S0_K12_SEED0',
    'single_cross_s0_k12_seed0_gate_summary.json',
)
X_PASS = x_summary['all_pass']
print()
print(f'X_PASS = {X_PASS}')


## 6. Cross-seed monotonicity verdict — k=12 seed=0 vs §8P1#2 k=12 anchor + §7.8' k=8 sister + 全 7-run 对照


In [ ]:
def read_baseline(root: Path, ref_final: float, ref_oob: float, ref_mean: float = None):
    fp = root / 'results' / 'final_eval.json'
    log = root / 'results' / 'eval_log.csv'
    out = {'final': ref_final, 'oob': ref_oob, 'mean': ref_mean, 'peak': None, 'peak_step': None, 'source': 'report'}
    if fp.exists():
        d = json.loads(fp.read_text(encoding='utf-8'))
        f = float(d['eval_success_rate'])
        c = d.get('eval_termination_counts', {})
        n = float(d.get('num_eval_episodes', EVAL_EPISODES))
        out['final'] = f
        out['oob'] = float(c.get('out_of_bounds', 0)) / max(n, 1.0)
        out['source'] = 'on-disk'
    if log.exists():
        dlog = pd.read_csv(log)
        out['mean'] = float(dlog['eval_success_rate'].mean())
        out['peak'] = float(dlog['eval_success_rate'].max())
        out['peak_step'] = int(dlog.loc[dlog['eval_success_rate'].idxmax(), 'env_step'])
    return out

print('=' * 110)
print('SINGLE_CROSS — k=12 cross-seed monotonicity verdict (seed=0 vs k=12 s42 anchor + k=8 s0 sister)')
print('-' * 110)

# 本 run (k=12 seed=0)
k12s0_final = float(x_summary['final_success_rate'])
k12s0_oob   = float(x_summary['final_oob_rate'])
k12s0_peak  = float(x_summary['peak_success_rate'])
k12s0_peak_step = int(x_summary['peak_step'])
k12s0_mean  = float(x_summary.get('mean_success_full_trajectory', 0.0))
k12s0_nsucc = int(x_summary.get('n_evals_with_success', 0))
k12s0_ntot  = int(x_summary.get('n_evals_total', 0))

# 7 个 baseline
b_k12_s42 = read_baseline(X_K12_S42_ROOT, 0.900, 0.100, 0.652)  # §8P1#2 anchor PASS-PLATEAU
b_k8_s42  = read_baseline(X_K8_S42_ROOT,  0.900, 0.100, 0.636)  # §7.8 anchor PASS
b_k8_s0   = read_baseline(X_K8_S0_ROOT,   0.500, 0.133, 0.260)  # §7.8' PARTIAL — 主对照
b_k8_s7   = read_baseline(X_K8_S7_ROOT,   0.867, 0.133, 0.518)  # §7.8'' BORDERLINE-PASS
b_k4_s42  = read_baseline(X_K4_S42_ROOT,  0.100, 0.667, 0.221)  # §7.6.4 floor
b_k4_s0   = read_baseline(X_K4_S0_ROOT,   0.400, 0.200, 0.218)  # §7.7.1 sister
b_s1      = read_baseline(X_S1_UPPER_ROOT, 0.900, 0.100, 0.497) # §7.1 upper

print()
print(f'{"config":<48}{"final":>10}{"mean":>10}{"oob":>10}{"peak":>10}{"peak@":>14}')
print('-' * 110)
label_k8_s7  = "vanilla s0_k8 s7  (§7.8'' BORDER)"
label_k8_s0  = "vanilla s0_k8 s0  (§7.8' PARTIAL)"
print(f'{"vanilla s1_k4 s42 (§7.1 upper ref)":<48}{b_s1["final"]:>10.4f}{(b_s1["mean"] or float("nan")):>10.4f}{b_s1["oob"]:>10.4f}{(b_s1["peak"] or float("nan")):>10.4f}{(b_s1["peak_step"] or 0):>14,}')
print(f'{"vanilla s0_k4 s42 (§7.6.4 FAIL floor)":<48}{b_k4_s42["final"]:>10.4f}{(b_k4_s42["mean"] or float("nan")):>10.4f}{b_k4_s42["oob"]:>10.4f}{(b_k4_s42["peak"] or float("nan")):>10.4f}{(b_k4_s42["peak_step"] or 0):>14,}')
print(f'{"vanilla s0_k4 s0  (§7.7.1 sister)":<48}{b_k4_s0["final"]:>10.4f}{(b_k4_s0["mean"] or float("nan")):>10.4f}{b_k4_s0["oob"]:>10.4f}{(b_k4_s0["peak"] or float("nan")):>10.4f}{(b_k4_s0["peak_step"] or 0):>14,}')
print(f'{"vanilla s0_k8 s42 (§7.8 anchor PASS)":<48}{b_k8_s42["final"]:>10.4f}{(b_k8_s42["mean"] or float("nan")):>10.4f}{b_k8_s42["oob"]:>10.4f}{(b_k8_s42["peak"] or float("nan")):>10.4f}{(b_k8_s42["peak_step"] or 0):>14,}')
print(f'{label_k8_s0:<48}{b_k8_s0["final"]:>10.4f}{(b_k8_s0["mean"] or float("nan")):>10.4f}{b_k8_s0["oob"]:>10.4f}{(b_k8_s0["peak"] or float("nan")):>10.4f}{(b_k8_s0["peak_step"] or 0):>14,}')
print(f'{label_k8_s7:<48}{b_k8_s7["final"]:>10.4f}{(b_k8_s7["mean"] or float("nan")):>10.4f}{b_k8_s7["oob"]:>10.4f}{(b_k8_s7["peak"] or float("nan")):>10.4f}{(b_k8_s7["peak_step"] or 0):>14,}')
print(f'{"vanilla s0_k12 s42 (§8P1#2 anchor PLAT)":<48}{b_k12_s42["final"]:>10.4f}{(b_k12_s42["mean"] or float("nan")):>10.4f}{b_k12_s42["oob"]:>10.4f}{(b_k12_s42["peak"] or float("nan")):>10.4f}{(b_k12_s42["peak_step"] or 0):>14,}')
print(f'{"vanilla s0_k12 s0  (THIS RUN, sister)":<48}{k12s0_final:>10.4f}{k12s0_mean:>10.4f}{k12s0_oob:>10.4f}{k12s0_peak:>10.4f}{k12s0_peak_step:>14,}')
print('=' * 110)

# Δ 行
delta_final_vs_k12_s42 = k12s0_final - b_k12_s42['final']
delta_mean_vs_k12_s42  = k12s0_mean  - (b_k12_s42['mean'] or 0.0)
delta_oob_vs_k12_s42   = k12s0_oob   - b_k12_s42['oob']
delta_final_vs_k8_s0   = k12s0_final - b_k8_s0['final']
delta_mean_vs_k8_s0    = k12s0_mean  - (b_k8_s0['mean'] or 0.0)
delta_oob_vs_k8_s0     = k12s0_oob   - b_k8_s0['oob']
delta_final_vs_s1      = k12s0_final - b_s1['final']

print()
print(f'{"contrast":<60}{"Δ final":>12}{"Δ mean":>12}{"Δ oob":>12}')
print('-' * 110)
print(f'{"k=12 s0 vs k=12 s42 (vs PASS-PLATEAU anchor)":<60}'
      f'{delta_final_vs_k12_s42:>+12.4f}{delta_mean_vs_k12_s42:>+12.4f}{delta_oob_vs_k12_s42:>+12.4f}')
print(f'{"k=12 s0 vs k=8 s0  (cross-history, same seed)":<60}'
      f'{delta_final_vs_k8_s0:>+12.4f}{delta_mean_vs_k8_s0:>+12.4f}{delta_oob_vs_k8_s0:>+12.4f}')
print(f'{"k=12 s0 vs s1_k4 s42 (gap-to-upper)":<60}'
      f'{delta_final_vs_s1:>+12.4f}'
      f'{(k12s0_mean - (b_s1["mean"] or 0)):>+12.4f}'
      f'{k12s0_oob - b_s1["oob"]:>+12.4f}')
print('=' * 110)
print()
print(f'k=12 seed=0 evals_with_success: {k12s0_nsucc} / {k12s0_ntot}  '
      f'(k=12 s42 was 35/39 PASS-PLATEAU, k=8 s0 was 32/39 PARTIAL, k=8 s7 was 31/39 BORDER)')

# Cross-seed monotonicity verdict — 5-tier
# (吸取 build_k8_seed7.py 4-tier 漏 BORDERLINE 教训 — 必须显式判 final≥0.85 但 OOB 卡线的情形)
strict_pass = (k12s0_final >= PASS_FINAL_SUCCESS) and (k12s0_oob <= PASS_OOB_RATE)
borderline_pass = (k12s0_final >= PASS_FINAL_SUCCESS) and (PASS_OOB_RATE < k12s0_oob <= BORDERLINE_OOB_RATE)
persistent_stall = (0.50 <= k12s0_final < PASS_FINAL_SUCCESS) and (abs(k12s0_final - b_k8_s0['final']) <= 0.20)
unexpected_decay = ((k12s0_final < b_k8_s0['final'] - 0.05) or (0.40 <= k12s0_final < 0.50))
collapse = k12s0_final < 0.40

# Priority order: strict_pass → borderline_pass → unexpected_decay → persistent_stall → collapse
# (unexpected_decay 优先于 persistent_stall 因为它是 cross-history regression 信号)
if strict_pass:
    verdict = (
        f'CROSS-SEED-RESCUE — k=12 s0 strict 5/5 PASS (final={k12s0_final:.3f}≥0.85, OOB={k12s0_oob:.3f}≤0.10); '
        f'与 k=12 s42 anchor (final=0.900) tight cluster, |Δfinal|={abs(delta_final_vs_k12_s42):.3f}; '
        f'cross-history vs k=8 s0 Δfinal=+{delta_final_vs_k8_s0:.3f} 大幅 rescue seed=0 stall. '
        f'强证据 H_rescue: history extension 不仅提升 lucky seed (s42)，还能 rescue unlucky seed (s0) local minimum. '
        f'§7.8 主张升格「k=4→8→12 monotonic; k=12 cross-seed robust 闭合 gap」; '
        f'§8 P1#2 升格「monotonic improvement extends to k=12 across seeds」; '
        f'下一步可选: 跑 k=8 seed=11 第 4 anchor 测 k=8 strict 边界，或 pivot §8 P0 variance reduction'
    )
elif borderline_pass:
    verdict = (
        f'CROSS-SEED-BORDERLINE-PASS — k=12 s0 final={k12s0_final:.3f}≥0.85 ✓ 但 OOB={k12s0_oob:.3f}>0.10 ✗ (≤0.135 BORDER); '
        f'final 维度 rescue 成立 (vs k=8 s0 Δ=+{delta_final_vs_k8_s0:.3f}); '
        f'但 OOB 仍 seed-dep noise 与 k=8 s7 BORDER (OOB=0.133) 一致; '
        f'SAC variance 主要表现在 OOB 末端而非 final goal 率. '
        f'§7.8 主张可升格「history extension cross-seed buys final rescue 但 OOB residual variance 不可忽略」; '
        f'§8 P0 variance reduction (DroQ / SimBa / N-Step) priority elevated — 用于压低 OOB tail 而非 final 提升'
    )
elif unexpected_decay:
    verdict = (
        f'UNEXPECTED-DECAY — k=12 s0 final={k12s0_final:.3f} <= k=8 s0 final={b_k8_s0["final"]:.3f}-0.05={b_k8_s0["final"]-0.05:.3f}; '
        f'over-stale history 在 seed=0 上反害, history extension 不再 monotonic. '
        f'§8 P1#2 主张回退「k=8 是 cross-seed sweet spot; k=12 仅在 lucky seed (s42) 有 mild speedup but seed=0 上 regress」; '
        f'强烈推荐 pivot §8 P0 variance reduction (DroQ / SimBa / N-Step) 代替 history extension'
    )
elif persistent_stall:
    verdict = (
        f'PERSISTENT-STALL — k=12 s0 final={k12s0_final:.3f} ∈ [0.5, 0.85), 与 k=8 s0 final={b_k8_s0["final"]:.3f} 在 0.20 范围内; '
        f'seed=0 stall 跨 history length 持续 (k=4: 0.40, k=8: 0.50, k=12: {k12s0_final:.3f}). '
        f'强证据 H_seed-stall: seed=0 有 init-dep local minimum 与 history length 弱相关; '
        f'§7.8 主张维持「k=4→8 在 2/3 seeds 闭合 gap; seed=0 stall 是 init-dep, 非 information bottleneck」; '
        f'§8 P0 variance reduction 必要 — 必须先解 seed-stall 才能再讨论 history extension'
    )
elif collapse:
    verdict = (
        f'COLLAPSE — k=12 s0 final={k12s0_final:.3f}<0.40, 严重下降. '
        f'必须 audit: trainer_state grad-norm / actor-critic loss curve / obs_dim 排除 training bug; '
        f'若非 bug，则 thesis claim 严重 reframe — history extension 不仅非 monotonic, 还在 unlucky seed 上 catastrophic'
    )
else:
    # 兜底分支 (理论上不应触发；保留以避免 catch-all bug 重现)
    verdict = (
        f'UNCLASSIFIED — final={k12s0_final:.3f} OOB={k12s0_oob:.3f}; '
        f'5-tier schema 未触发任何明确条件; 需 manual 审视 (本 verdict 应被视为 bug 报告 — 类似 §7.8'' seed=7 误判教训)'
    )

print()
print(f'>>> verdict: {verdict}')

# 落盘 cross-seed monotonicity summary
mono_out = {
    'experiment': 'arrival_v2_s0_cross_k12_seed0_crossseed_monotonicity',
    'seed': SEED,
    'benchmark': X_BENCHMARK_KEY,
    'probe_layout': PROBE_LAYOUT,
    'history_length': HISTORY_LENGTH,
    'total_steps': X_TOTAL_STEPS,
    'algorithm': 'sac_vanilla',
    'cli_diff_vs_§8P1#2_anchor': '--seed 42 → 0',
    'results': {
        'k12_s0_thisrun': {
            'final': k12s0_final, 'mean': k12s0_mean, 'oob': k12s0_oob,
            'peak': k12s0_peak, 'peak_step': k12s0_peak_step,
            'n_evals_with_success': k12s0_nsucc, 'n_evals_total': k12s0_ntot,
        },
        'k12_s42_anchor_§8P1#2_PASS_PLATEAU': b_k12_s42,
        'k8_s42_anchor_§7.8_PASS': b_k8_s42,
        "k8_s0_§7.8'_PARTIAL_main_contrast": b_k8_s0,
        "k8_s7_§7.8''_BORDERLINE_PASS": b_k8_s7,
        '§7.6.4_k4_s42_floor': b_k4_s42,
        '§7.7.1_k4_s0_sister': b_k4_s0,
        '§7.1_s1_k4_s42_upper': b_s1,
    },
    'delta_vs_§8P1#2_k12_anchor_seed42': {
        'final_pp': round(delta_final_vs_k12_s42 * 100, 2),
        'mean_pp':  round(delta_mean_vs_k12_s42  * 100, 2),
        'oob_pp':   round(delta_oob_vs_k12_s42   * 100, 2),
    },
    "delta_vs_§7.8'_k8_seed0": {
        'final_pp': round(delta_final_vs_k8_s0 * 100, 2),
        'mean_pp':  round(delta_mean_vs_k8_s0  * 100, 2),
        'oob_pp':   round(delta_oob_vs_k8_s0   * 100, 2),
    },
    'seed0_cross_history_trajectory': {
        'k4':  b_k4_s0['final'],
        'k8':  b_k8_s0['final'],
        'k12': k12s0_final,
    },
    'verdict_tier': (
        'CROSS-SEED-RESCUE' if strict_pass
        else 'CROSS-SEED-BORDERLINE-PASS' if borderline_pass
        else 'UNEXPECTED-DECAY' if unexpected_decay
        else 'PERSISTENT-STALL' if persistent_stall
        else 'COLLAPSE' if collapse
        else 'UNCLASSIFIED'
    ),
    'all_pass': bool(x_summary['all_pass']),
    'verdict': verdict,
}
out_dir = Path('experiments/arrival_v2_prototype/s0_cross_k12_seed0_summary')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'combined_gate_summary.json'
out_path.write_text(json.dumps(mono_out, indent=2), encoding='utf-8')
print(f'\n[saved] {out_path}')
